In [1]:
import os
os.environ.pop("ALL_PROXY", None)
os.environ.pop("all_proxy", None)

'socks://127.0.0.1:7897/'

In [2]:
import torch
from PIL import Image
import open_clip
from imagenetv2_pytorch import ImageNetV2Dataset
from torch.utils.data import DataLoader

In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
# Load the model and run the model on a single image 
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active
tokenizer = open_clip.get_tokenizer('ViT-B-16')

image = preprocess(Image.open("../data/CLIP.png")).unsqueeze(0)
text = tokenizer(["a diagram", "a dog", "a cat"])

with torch.no_grad(), torch.autocast("cpu"):
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

print("Label probs:", text_probs)  # prints: [[1., 0., 0.]]

/home/lcoach/research/vlm-reliability/.venv/lib/python3.12/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Label probs: tensor([[0.6680, 0.3164, 0.0157]], dtype=torch.bfloat16)


### Load the ImageNet-V2 dataset

In [5]:
dataset = ImageNetV2Dataset("matched-frequency", location= "../data", transform=preprocess) # supports matched-frequency, threshold-0.7, top-images variants
dataloader = DataLoader(dataset, batch_size=32) # use whatever batch size you wish

In [6]:
for batch_idx, (image, label) in enumerate(dataloader):
    print(f"Batch ID: {batch_idx} Image Shape: {image.shape}, Label Shape: {label.shape}")
    break

Batch ID: 0 Image Shape: torch.Size([32, 3, 224, 224]), Label Shape: torch.Size([32])


### Build zero-shot text classifier from ImageNet class names

In [8]:
import json
from torchvision.datasets.utils import download_url

# Download the official ImageNet class index mapping
download_url(
    "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json",
    "../data",
    "imagenet_class_index.json",
)

# Load the JSON mapping file
with open("../data/imagenet_class_index.json", "r") as f:
    class_idx = json.load(f)

# Convert to a list where index 0-999 corresponds to model outputs
class_names = [class_idx[str(i)][1] for i in range(1000)]

# Example: Print class name for index 263
print(class_names[263])  # Output: Pembroke (welsh_corgi)

Pembroke


In [9]:
len(class_names)

1000

In [10]:
prompts = []

for cls in class_names:
  clean_cls = cls.replace("_", " ")
  prompts.append(f"a photo of a {clean_cls}")

In [11]:
prompts[:10]

['a photo of a tench',
 'a photo of a goldfish',
 'a photo of a great white shark',
 'a photo of a tiger shark',
 'a photo of a hammerhead',
 'a photo of a electric ray',
 'a photo of a stingray',
 'a photo of a cock',
 'a photo of a hen',
 'a photo of a ostrich']

In [ ]:
model.to(device)

In [12]:
text_tokens = tokenizer(prompts)
text_tokens.shape

torch.Size([1000, 77])

In [ ]:
with torch.no_grad():
    text_tokens = text_tokens.to(device)
    text_features = model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)

text_features.shape

torch.Size([1000, 512])

In [ ]:
# Cache the text features to avoid recomputing them every time
#torch.save(text_features.cpu(), "features/imagenetv2_text_features.pt")

### Compute and cache the image features

In [ ]:
all_features = []
all_labels = []

for images, labels in dataloader:
    images = images.to(device)

    with torch.no_grad():
        feats = model.encode_image(images)
        feats /= feats.norm(dim=-1, keepdim=True)
    
    all_features.append(feats.cpu())
    all_labels.append(labels)

all_features = torch.cat(all_features)
all_labels = torch.cat(all_labels)     

torch.save({"features": all_features, "labels": all_labels}, "features/imagenetv2.pt")

### Zero Shot Eval Loop - First Attempt

In [ ]:
  ### First attempt — 52.68% (bug: underscores in class names, e.g. "great_white_shark")

# correct = 0
# total = 0

# for batch_idx, (images, labels) in enumerate(dataloader):
    
#     with torch.no_grad(), torch.autocast("cuda"):
#         image_features = model.encode_image(images)
#         image_features /= image_features.norm(dim=-1, keepdim=True)

#         similarity = image_features @ text_features.T 
#         preds = similarity.argmax(dim=1)

#         correct += (preds == labels).sum().item()
#         total += labels.size(0)

#     print(f"Current batch : {batch_idx + 1}")


# accuracy = correct / total
# print(f"Accuracy: {accuracy:.2%}")

Accuracy: 59.38%


### Zero Shot Evaluation

In [ ]:
cached = torch.load("features/imagenetv2.pt")
image_features = cached["features"]
labels = cached["labels"]
text_features = torch.load("features/imagenetv2_text_features.pt")

similarity = image_features @ text_features.T
preds = similarity.argmax(dim=1)
accuracy = (preds == labels).float().mean().item()

In [ ]:
print(f"Accuracy: {accuracy:.2%}")